# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.ROUTES = Set(initialize=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
model.NOEUDS = Set(initialize=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I'])
model.ARC = Set(dimen=2, initialize=[(i0,i1) for i0 in model.NOEUDS for i1 in model.ROUTES])

## 🔹 Parameters

In [ ]:
model.Temps = Param(model.ROUTES, initialize={1: 6.0, 2: 4.0, 3: 7.0, 4: 5.0, 5: 4.0, 6: 6.0, 7: 5.0, 8: 3.0, 9: 7.0, 10: 6.0}, within=NonNegativeReals)
model.CONNEXION = Param(model.NOEUDS, model.ROUTES, initialize={('A', 1): 1.0, ('A', 2): 0.0, ('A', 3): 0.0, ('A', 4): 0.0, ('A', 5): 1.0, ('A', 6): 0.0, ('A', 7): 0.0, ('A', 8): 0.0, ('A', 9): 1.0, ('A', 10): 0.0, ('B', 1): 0.0, ('B', 2): 1.0, ('B', 3): 0.0, ('B', 4): 1.0, ('B', 5): 0.0, ('B', 6): 1.0, ('B', 7): 0.0, ('B', 8): 0.0, ('B', 9): 1.0, ('B', 10): 1.0, ('C', 1): 0.0, ('C', 2): 0.0, ('C', 3): 1.0, ('C', 4): 1.0, ('C', 5): 0.0, ('C', 6): 0.0, ('C', 7): 1.0, ('C', 8): 0.0, ('C', 9): 1.0, ('C', 10): 0.0, ('D', 1): 1.0, ('D', 2): 0.0, ('D', 3): 0.0, ('D', 4): 0.0, ('D', 5): 0.0, ('D', 6): 1.0, ('D', 7): 0.0, ('D', 8): 1.0, ('D', 9): 0.0, ('D', 10): 0.0, ('E', 1): 0.0, ('E', 2): 0.0, ('E', 3): 1.0, ('E', 4): 1.0, ('E', 5): 0.0, ('E', 6): 1.0, ('E', 7): 0.0, ('E', 8): 0.0, ('E', 9): 0.0, ('E', 10): 0.0, ('F', 1): 0.0, ('F', 2): 1.0, ('F', 3): 0.0, ('F', 4): 0.0, ('F', 5): 1.0, ('F', 6): 0.0, ('F', 7): 0.0, ('F', 8): 0.0, ('F', 9): 0.0, ('F', 10): 0.0, ('G', 1): 1.0, ('G', 2): 0.0, ('G', 3): 0.0, ('G', 4): 0.0, ('G', 5): 0.0, ('G', 6): 0.0, ('G', 7): 1.0, ('G', 8): 1.0, ('G', 9): 0.0, ('G', 10): 1.0, ('H', 1): 0.0, ('H', 2): 0.0, ('H', 3): 1.0, ('H', 4): 0.0, ('H', 5): 1.0, ('H', 6): 0.0, ('H', 7): 0.0, ('H', 8): 0.0, ('H', 9): 0.0, ('H', 10): 1.0, ('I', 1): 0.0, ('I', 2): 1.0, ('I', 3): 0.0, ('I', 4): 1.0, ('I', 5): 0.0, ('I', 6): 0.0, ('I', 7): 1.0, ('I', 8): 0.0, ('I', 9): 0.0, ('I', 10): 0.0}, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.ROUTES, domain=Binary)

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=sum(model.X[r] for r in model.ROUTES) == 3)
model.c_for_0 = ConstraintList()
for n in model.NOEUDS:
    model.c_for_0.add(sum(model.CONNEXION[n, r] * model.X[r] for r in model.ROUTES) == 1)
# @BIN/@GIN directive already handled in variable declarations

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.Temps[r] * model.X[r] for r in model.ROUTES), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')